# 🧬 Tissue Classification - Colab Ready

**Self-contained notebook** - All modules embedded below

Generated from `TRAIN.ipynb` template with modules injected

---

In [ ]:
# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Change to your data directory
    import os
    os.chdir('/content/drive/MyDrive/Challenge2')
    print(f"✅ Working directory: {os.getcwd()}")
except:
    print("ℹ️  Not running in Colab, using current directory")

In [ ]:
# Install dependencies
!pip install -q pyyaml tensorboard scikit-learn tqdm
print('✅ Dependencies installed')

In [ ]:
#@title 📝 config_optimized.yaml (click to expand)
config_yaml_content = r'''# ============================================
# OPTIMIZED CONFIGURATION FOR TISSUE CLASSIFICATION
# Implements all performance improvements from analysis
# Expected F1: 0.93-0.96 (vs baseline 0.85-0.92)
# ============================================

# ========== REPRODUCIBILITY ==========
SEED: 42
EXPERIMENT_NAME: efficientnetv2_tissue_optimized

# ========== PATHS ==========
DATA_DIR: ./train_data
TEST_DIR: ./test_data
CSV_PATH: ./train_labels.csv  # or clean_train_labels.csv if using outlier detection
OUTPUT_DIR: ./outputs
MODELS_DIR: ./models_optimized
CHECKPOINT_DIR: ./checkpoints
TENSORBOARD_DIR: ./tensorboard
VISUALIZATION_DIR: ./visualizations

# ========== DATA ==========
IMG_SIZE: 384  # Larger for EfficientNet (captures more tissue detail)
VAL_SPLIT: 0.15  # Smaller validation = more training data
BATCH_SIZE: 16  # Smaller due to larger images
NUM_WORKERS: 4
PIN_MEMORY: true
PREFETCH_FACTOR: 4  # Increased for better GPU utilization

# ========== MODEL ==========
MODEL_NAME: efficientnet_v2_s  # Best accuracy/speed for medical imaging
USE_PRETRAINED: true
FREEZE_BACKBONE: false
USE_ATTENTION: true  # Critical for tissue localization
DROPOUT: 0.3

# ========== TRAINING ==========
NUM_EPOCHS: 100
LEARNING_RATE: 0.0003  # Lion optimizer needs lower LR
WEIGHT_DECAY: 0.03  # Lion needs higher weight decay
OPTIMIZER: lion  # Modern optimizer, memory-efficient

# ⚖️ Loss function - CRITICAL for class imbalance
USE_WEIGHTED_LOSS: true  # Addresses Triple negative underrepresentation
CLASS_WEIGHTS: [2.85, 1.08, 1.0, 1.12]  # [Triple neg: 156, Luminal A: 414, Luminal B: 445, HER2+: 397]
USE_FOCAL_LOSS: false  # Alternative to weighted loss (use one or the other)
FOCAL_ALPHA: 1.0
FOCAL_GAMMA: 2.0

# Scheduler
USE_SCHEDULER: true
SCHEDULER: cosine  # Better than plateau for fixed epochs
PATIENCE_SCHEDULER: 5
FACTOR: 0.5

# Early stopping
PATIENCE: 15  # More patience for better convergence
MONITOR_METRIC: val_f1

# Checkpointing
CHECKPOINT_INTERVAL: 5
SAVE_BEST_ONLY: true

# Advanced
USE_MIXED_PRECISION: true  # 2x faster, 40% less memory
GRADIENT_CLIP: 1.0  # Prevent exploding gradients
LABEL_SMOOTHING: 0.0

# Progressive Training (optional - train in 2 phases)
USE_PROGRESSIVE_TRAINING: false  # Set true for 2-phase training
FREEZE_EPOCHS: 10
UNFREEZE_LR_MULTIPLIER: 0.1

# ========== AUGMENTATION ==========
USE_AUGMENTATION: true

# 🤖 Automated Augmentation - RECOMMENDED
USE_AUTOMATED_AUG: true  # Let the machine find optimal policy
AUTOMATED_AUG_METHOD: randaugment
RANDAUGMENT_N: 2  # Number of operations
RANDAUGMENT_M: 7  # Moderate magnitude for medical images

# Manual Augmentation (fallback if USE_AUTOMATED_AUG=false)
HORIZONTAL_FLIP: 0.5
VERTICAL_FLIP: 0.5
ROTATION_DEGREES: 15
COLOR_JITTER: 0.2
RANDOM_ERASING: 0.0

# ✂️ Advanced Augmentation - High Impact
USE_MIXUP: false  # CutMix is better for tissue images
MIXUP_ALPHA: 0.2
MIXUP_PROB: 0.5
USE_CUTMIX: true  # Recommended: +2-3% F1
CUTMIX_ALPHA: 1.0
CUTMIX_PROB: 0.5

# 🔄 Test-Time Augmentation (TTA) - Use for final submission
USE_TTA: false  # Set true for submission (+1-2% accuracy, but 5x slower)
TTA_NUM_AUGMENTATIONS: 5

# ========== NORMALIZATION ==========
USE_IMAGENET_NORM: true  # MANDATORY for pretrained models
NORM_TYPE: group  # Better than batch for small batch sizes (<16)

# ========== VISUALIZATION ==========
VISUALIZE_ATTENTION: false
VISUALIZE_ACTIVATIONS: false
SAVE_ATTENTION_MAPS: false

# ========== TENSORBOARD ==========
USE_TENSORBOARD: true
LOG_IMAGES: true
LOG_HISTOGRAMS: false
LOG_INTERVAL: 50
'''

# Write config to file
with open('config.yaml', 'w') as f:
    f.write(config_yaml_content)

print('✅ config.yaml created (from config_optimized.yaml)')

In [ ]:
#@title 📦 Module Files (click to expand)
# All modules embedded - no imports needed


# ============================================================
# config.py
# ============================================================

"""
Single source of truth for all configuration
ALL VALUES MUST COME FROM YAML - no hardcoded defaults
"""
import os
import random
from pathlib import Path

try:
    import yaml
    YAML_AVAILABLE = True
except ImportError:
    YAML_AVAILABLE = False
    print("ERROR: PyYAML is required. Install with: pip install pyyaml")


class Config:
    """Configuration loaded from YAML file only - no defaults in code"""
    
    def __init__(self, yaml_path='config.yaml'):
        """Load all configuration from YAML file
        
        Args:
            yaml_path: Path to YAML config file (required)
        """
        if not YAML_AVAILABLE:
            raise ImportError("PyYAML required. Install: pip install pyyaml")
        
        yaml_path = Path(yaml_path)
        if not yaml_path.exists():
            raise FileNotFoundError(
                f"Config file not found: {yaml_path}\n"
                f"Please create config.yaml or specify path to existing config."
            )
        
        # Load YAML
        with open(yaml_path, 'r') as f:
            config_dict = yaml.safe_load(f)
        
        if not config_dict:
            raise ValueError(f"Empty config file: {yaml_path}")
        
        # Set all attributes from YAML
        for key, value in config_dict.items():
            setattr(self, key, value)
        
        # Validate required keys
        self._validate()
        
        print(f"✅ Config loaded from {yaml_path}")
        print(f"   {len(config_dict)} parameters loaded")
    
    def _validate(self):
        """Validate that required config keys are present"""
        required = [
            # Reproducibility
            'SEED', 'EXPERIMENT_NAME',
            # Paths
            'DATA_DIR', 'CSV_PATH', 'TEST_DIR', 'OUTPUT_DIR', 'MODELS_DIR',
            'CHECKPOINT_DIR', 'TENSORBOARD_DIR', 'VISUALIZATION_DIR',
            # Data
            'IMG_SIZE', 'VAL_SPLIT', 'BATCH_SIZE', 'NUM_WORKERS', 'PIN_MEMORY', 'PREFETCH_FACTOR',
            # Model (NUM_CLASSES is auto-detected, not required in YAML)
            'MODEL_NAME', 'USE_PRETRAINED', 'FREEZE_BACKBONE', 'USE_ATTENTION', 'DROPOUT',
            # Training
            'NUM_EPOCHS', 'LEARNING_RATE', 'WEIGHT_DECAY', 'OPTIMIZER',
            # Loss function
            'USE_WEIGHTED_LOSS', 'CLASS_WEIGHTS', 'USE_FOCAL_LOSS', 'FOCAL_ALPHA', 'FOCAL_GAMMA',
            # Scheduler
            'USE_SCHEDULER', 'SCHEDULER', 'PATIENCE_SCHEDULER', 'FACTOR',
            # Early stopping & checkpointing
            'PATIENCE', 'MONITOR_METRIC', 'CHECKPOINT_INTERVAL', 'SAVE_BEST_ONLY',
            # Advanced training
            'USE_MIXED_PRECISION', 'GRADIENT_CLIP', 'LABEL_SMOOTHING',
            'USE_PROGRESSIVE_TRAINING', 'FREEZE_EPOCHS', 'UNFREEZE_LR_MULTIPLIER',
            # Augmentation
            'USE_AUGMENTATION', 'USE_AUTOMATED_AUG', 'AUTOMATED_AUG_METHOD',
            'RANDAUGMENT_N', 'RANDAUGMENT_M',
            'HORIZONTAL_FLIP', 'VERTICAL_FLIP', 'ROTATION_DEGREES', 'COLOR_JITTER', 'RANDOM_ERASING',
            'USE_MIXUP', 'MIXUP_ALPHA', 'MIXUP_PROB', 
            'USE_CUTMIX', 'CUTMIX_ALPHA', 'CUTMIX_PROB',
            'USE_TTA', 'TTA_NUM_AUGMENTATIONS',
            # Normalization
            'USE_IMAGENET_NORM', 'NORM_TYPE',
            # Visualization
            'VISUALIZE_ATTENTION', 'VISUALIZE_ACTIVATIONS', 'SAVE_ATTENTION_MAPS',
            # TensorBoard
            'USE_TENSORBOARD', 'LOG_IMAGES', 'LOG_HISTOGRAMS', 'LOG_INTERVAL'
        ]
        
        missing = [k for k in required if not hasattr(self, k)]
        if missing:
            raise ValueError(
                f"Missing required config keys: {missing}\n"
                f"Please add them to your config.yaml\n"
                f"See config.yaml for complete template with all required fields."
            )
    
    @classmethod
    def from_yaml(cls, yaml_path):
        """Load config from YAML file (alias for __init__)"""
        return cls(yaml_path)
    
    def save_yaml(self, yaml_path):
        """Save current config to YAML file"""
        config_dict = {k: v for k, v in vars(self).items() 
                      if not k.startswith('_')}
        
        yaml_path = Path(yaml_path)
        yaml_path.parent.mkdir(parents=True, exist_ok=True)
        
        with open(yaml_path, 'w') as f:
            yaml.dump(config_dict, f, default_flow_style=False, sort_keys=False)
        
        print(f"✅ Config saved to {yaml_path}")
    
    def __repr__(self):
        """Pretty print config"""
        items = [f"{k}={v}" for k, v in vars(self).items() 
                if not k.startswith('_')]
        return f"Config({len(items)} params)"


def set_seed(seed):
    """Set all random seeds for reproducibility"""
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except ImportError:
        pass


def setup_device():
    """Setup and return the best available device"""
    try:
        import torch
        if torch.cuda.is_available():
            device = torch.device("cuda")
            print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
            print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        else:
            device = torch.device("cpu")
            print("⚠️  Using CPU (training will be slow)")
        return device
    except ImportError:
        print("⚠️  PyTorch not installed, assuming CPU")
        return None


def create_dirs(config):
    """Create necessary directories from config"""
    dirs = []
    for attr in ['OUTPUT_DIR', 'MODELS_DIR', 'CHECKPOINT_DIR', 
                 'TENSORBOARD_DIR', 'VISUALIZATION_DIR']:
        if hasattr(config, attr):
            dirs.append(getattr(config, attr))
    
    for dir_path in dirs:
        os.makedirs(dir_path, exist_ok=True)




# ============================================================
# automated_augmentation.py
# ============================================================

"""
Automated augmentation strategies
Implements ADVICE 06/12 - Automated Augmentation
"""
import torch
import torch.nn as nn
from torchvision import transforms
import random
import numpy as np


class RandAugment:
    """
    RandAugment: Practical automated data augmentation
    Paper: https://arxiv.org/abs/1909.13719
    
    Randomly select N augmentation transformations from a set and apply them
    with magnitude M.
    """
    def __init__(self, n=2, m=9):
        """
        Args:
            n: Number of augmentation transformations to apply sequentially
            m: Magnitude for all augmentation operations (0-10 scale)
        """
        self.n = n
        self.m = m
        self.augment_list = [
            self.auto_contrast,
            self.equalize,
            self.rotate,
            self.solarize,
            self.color,
            self.posterize,
            self.contrast,
            self.brightness,
            self.sharpness,
            self.shear_x,
            self.shear_y,
            self.translate_x,
            self.translate_y
        ]
    
    def __call__(self, img):
        """Apply N random augmentations"""
        ops = random.choices(self.augment_list, k=self.n)
        for op in ops:
            img = op(img)
        return img
    
    def auto_contrast(self, img):
        return transforms.functional.autocontrast(img)
    
    def equalize(self, img):
        return transforms.functional.equalize(img)
    
    def rotate(self, img):
        magnitude = (self.m / 10) * 30  # Max 30 degrees
        angle = random.uniform(-magnitude, magnitude)
        return transforms.functional.rotate(img, angle)
    
    def solarize(self, img):
        magnitude = int((self.m / 10) * 256)
        return transforms.functional.solarize(img, threshold=256 - magnitude)
    
    def color(self, img):
        magnitude = (self.m / 10) * 0.9 + 0.1  # 0.1 to 1.0
        return transforms.functional.adjust_saturation(img, magnitude)
    
    def posterize(self, img):
        magnitude = int((self.m / 10) * 4)  # Reduce bits by 0-4
        bits = 8 - magnitude
        return transforms.functional.posterize(img, bits)
    
    def contrast(self, img):
        magnitude = (self.m / 10) * 0.9 + 0.1
        return transforms.functional.adjust_contrast(img, magnitude)
    
    def brightness(self, img):
        magnitude = (self.m / 10) * 0.9 + 0.1
        return transforms.functional.adjust_brightness(img, magnitude)
    
    def sharpness(self, img):
        magnitude = (self.m / 10) * 0.9 + 0.1
        return transforms.functional.adjust_sharpness(img, magnitude)
    
    def shear_x(self, img):
        magnitude = (self.m / 10) * 0.3  # Max shear 0.3
        return transforms.functional.affine(img, angle=0, translate=[0, 0], 
                                           scale=1, shear=[magnitude * 180 / np.pi, 0])
    
    def shear_y(self, img):
        magnitude = (self.m / 10) * 0.3
        return transforms.functional.affine(img, angle=0, translate=[0, 0], 
                                           scale=1, shear=[0, magnitude * 180 / np.pi])
    
    def translate_x(self, img):
        magnitude = int((self.m / 10) * img.size[0] * 0.3)  # Max 30% of width
        return transforms.functional.affine(img, angle=0, translate=[magnitude, 0], 
                                           scale=1, shear=[0, 0])
    
    def translate_y(self, img):
        magnitude = int((self.m / 10) * img.size[1] * 0.3)
        return transforms.functional.affine(img, angle=0, translate=[0, magnitude], 
                                           scale=1, shear=[0, 0])


class TrivialAugmentWide:
    """
    TrivialAugment: Simple and effective augmentation
    Paper: https://arxiv.org/abs/2103.10158
    
    Randomly selects ONE augmentation per image with random magnitude.
    Often outperforms more complex strategies.
    """
    def __init__(self):
        self.augment_list = [
            ('Identity', 0, 0),
            ('AutoContrast', 0, 0),
            ('Equalize', 0, 0),
            ('Rotate', -30, 30),
            ('Solarize', 0, 256),
            ('Color', 0.1, 1.9),
            ('Posterize', 2, 8),
            ('Contrast', 0.1, 1.9),
            ('Brightness', 0.1, 1.9),
            ('Sharpness', 0.1, 1.9),
            ('ShearX', -0.3, 0.3),
            ('ShearY', -0.3, 0.3),
            ('TranslateX', -0.3, 0.3),
            ('TranslateY', -0.3, 0.3)
        ]
    
    def __call__(self, img):
        """Randomly select and apply ONE augmentation"""
        op_name, min_val, max_val = random.choice(self.augment_list)
        magnitude = random.uniform(min_val, max_val) if max_val > min_val else 0
        
        if op_name == 'Identity':
            return img
        elif op_name == 'AutoContrast':
            return transforms.functional.autocontrast(img)
        elif op_name == 'Equalize':
            return transforms.functional.equalize(img)
        elif op_name == 'Rotate':
            return transforms.functional.rotate(img, magnitude)
        elif op_name == 'Solarize':
            return transforms.functional.solarize(img, int(magnitude))
        elif op_name == 'Color':
            return transforms.functional.adjust_saturation(img, magnitude)
        elif op_name == 'Posterize':
            return transforms.functional.posterize(img, int(magnitude))
        elif op_name == 'Contrast':
            return transforms.functional.adjust_contrast(img, magnitude)
        elif op_name == 'Brightness':
            return transforms.functional.adjust_brightness(img, magnitude)
        elif op_name == 'Sharpness':
            return transforms.functional.adjust_sharpness(img, magnitude)
        elif op_name == 'ShearX':
            return transforms.functional.affine(img, angle=0, translate=[0, 0], 
                                               scale=1, shear=[magnitude * 180 / np.pi, 0])
        elif op_name == 'ShearY':
            return transforms.functional.affine(img, angle=0, translate=[0, 0], 
                                               scale=1, shear=[0, magnitude * 180 / np.pi])
        elif op_name == 'TranslateX':
            pixels = int(magnitude * img.size[0])
            return transforms.functional.affine(img, angle=0, translate=[pixels, 0], 
                                               scale=1, shear=[0, 0])
        elif op_name == 'TranslateY':
            pixels = int(magnitude * img.size[1])
            return transforms.functional.affine(img, angle=0, translate=[0, pixels], 
                                               scale=1, shear=[0, 0])
        return img


def get_automated_augmentation(method='randaugment', **kwargs):
    """
    Factory function to get automated augmentation
    
    Args:
        method: 'randaugment', 'trivialaugment', or 'autoaugment'
        **kwargs: Method-specific parameters
    
    Returns:
        Augmentation transform
    """
    if method == 'randaugment':
        n = kwargs.get('n', 2)  # Number of ops
        m = kwargs.get('m', 9)  # Magnitude
        return RandAugment(n=n, m=m)
    
    elif method == 'trivialaugment':
        return TrivialAugmentWide()
    
    elif method == 'autoaugment':
        # Use PyTorch's built-in AutoAugment
        from torchvision.transforms import AutoAugment, AutoAugmentPolicy
        policy = kwargs.get('policy', AutoAugmentPolicy.IMAGENET)
        return AutoAugment(policy=policy)
    
    else:
        raise ValueError(f"Method {method} not supported. Choose from: "
                        "randaugment, trivialaugment, autoaugment")



# ============================================================
# dataset.py
# ============================================================

"""
Custom Dataset and DataLoader utilities
"""
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ImageNet normalization statistics (critical for pretrained models)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class TissueDataset(Dataset):
    """Dataset for tissue classification"""
    
    def __init__(self, df, data_dir, transform=None, use_mask=True):
        """
        Args:
            df: DataFrame with Image, Mask, Label columns
            data_dir: Root directory containing images/ and masks/ folders
            transform: Optional transforms
            use_mask: Whether to apply mask to images
        """
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.transform = transform
        self.use_mask = use_mask
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Handle both uppercase and lowercase column names
        img_name = row.get('Image', row.get('sample_index', row.get('image')))
        mask_name = row.get('Mask', row.get('mask', img_name))  # Use same name if mask col doesn't exist
        label_val = row.get('Label', row.get('label'))
        
        # Load image and mask - check if images/ subdirectory exists
        img_subdir_path = os.path.join(self.data_dir, "images", img_name)
        img_direct_path = os.path.join(self.data_dir, img_name)
        
        if os.path.exists(img_subdir_path):
            img_path = img_subdir_path
            mask_path = os.path.join(self.data_dir, "masks", mask_name)
        else:
            img_path = img_direct_path
            mask_path = img_direct_path  # Same file serves as both image and mask
        
        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')
        
        # Apply mask if needed
        if self.use_mask:
            image_np = np.array(image)
            mask_np = np.array(mask)
            
            # Instead of setting background to black (0), set to ImageNet mean
            # This prevents distribution shift when masks aren't used
            imagenet_mean_rgb = np.array([0.485 * 255, 0.456 * 255, 0.406 * 255], dtype=np.uint8)
            
            # Create background with ImageNet mean color
            masked_image = np.where(
                mask_np[:, :, np.newaxis] > 0,  # Where mask is active
                image_np,  # Keep original image
                imagenet_mean_rgb  # Use ImageNet mean for background
            )
            image = Image.fromarray(masked_image.astype(np.uint8))
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        # Handle label (convert string to int if needed)
        if isinstance(label_val, str):
            # Map class names to integers (handle both HER2(+) and HER2 positive)
            class_mapping = {
                'Triple negative': 0,
                'Luminal A': 1,
                'Luminal B': 2,
                'HER2 positive': 3,
                'HER2(+)': 3  # Alternative format in dataset
            }
            label = torch.tensor(class_mapping.get(label_val, 0), dtype=torch.long)
        else:
            label = torch.tensor(label_val, dtype=torch.long)
        
        return image, label


def get_transforms(img_size=256, augment=True, use_imagenet_norm=True,
                   use_automated_aug=False, auto_method='randaugment', auto_n=2, auto_m=9):
    """Get train and validation transforms
    
    Args:
        img_size: Target image size
        augment: Whether to apply data augmentation (training)
        use_imagenet_norm: Use ImageNet normalization (required for pretrained models)
        use_automated_aug: Use automated augmentation (RandAugment, TrivialAugment)
        auto_method: Automated augmentation method ('randaugment' or 'trivialaugment')
        auto_n: Number of augmentation operations (for RandAugment)
        auto_m: Magnitude of augmentations 0-10 (for RandAugment)
    
    Uses manual augmentation from original notebooks (challenge-2-preprocessing.ipynb):
    - ColorJitter, RandomRotation, RandomHorizontalFlip, RandomErasing
    Or automated augmentation from ADVICE 06/12
    """
    
    # Normalization (ImageNet stats for transfer learning - from lectures)
    normalize = transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    ) if use_imagenet_norm else transforms.Lambda(lambda x: x)
    
    if augment:
        if use_automated_aug:
            # Automated augmentation - ADVICE 06/12: "Let the policy emerge from the struggle"
            # get_automated_augmentation imported at module level
            auto_aug = get_automated_augmentation(method=auto_method, n=auto_n, m=auto_m)
            
            transform_list = [
                transforms.Resize((img_size, img_size)),
                auto_aug,
                transforms.ToTensor(),
                normalize
            ]
        else:
            # Manual augmentation from original challenge-2-preprocessing.ipynb
            transform_list = [
                transforms.Resize((img_size, img_size)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.5),
                transforms.RandomAffine(
                    degrees=15,                # Rotation
                    translate=(0.1, 0.1),      # Translation
                    scale=(0.9, 1.1)           # Zoom
                ),
                transforms.ColorJitter(
                    brightness=0.2, 
                    contrast=0.2, 
                    saturation=0.2
                ),
                transforms.ToTensor(),
                transforms.RandomErasing(
                    p=0.3,                     # Probability
                    scale=(0.02, 0.15),        # Size of erased area
                    ratio=(0.3, 3.3),          # Aspect ratio
                    value='random'             # Fill with random values
                ),
                normalize
            ]
        
        train_transform = transforms.Compose(transform_list)
    else:
        train_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            normalize
        ])
    
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        normalize
    ])
    
    return train_transform, val_transform


def create_dataloaders(train_df, val_df, data_dir, config, use_imagenet_norm=True):
    """Create optimized train and validation dataloaders
    
    Args:
        train_df: Training DataFrame
        val_df: Validation DataFrame
        data_dir: Root data directory
        config: Configuration object
        use_imagenet_norm: Use ImageNet normalization (True for pretrained models)
    
    Returns:
        train_loader, val_loader, num_classes
    """
    
    # Check for automated augmentation settings
    use_automated_aug = getattr(config, 'USE_AUTOMATED_AUG')
    auto_method = getattr(config, 'AUTOMATED_AUG_METHOD')
    auto_n = getattr(config, 'RANDAUGMENT_N')
    auto_m = getattr(config, 'RANDAUGMENT_M')
    
    train_transform, val_transform = get_transforms(
        img_size=config.IMG_SIZE,
        augment=True,
        use_imagenet_norm=use_imagenet_norm,
        use_automated_aug=use_automated_aug,
        auto_method=auto_method,
        auto_n=auto_n,
        auto_m=auto_m
    )
    
    train_dataset = TissueDataset(train_df, data_dir, transform=train_transform)
    val_dataset = TissueDataset(val_df, data_dir, transform=val_transform)
    
    # Auto-detect number of classes from data
    label_col = 'label' if 'label' in train_df.columns else 'Label'
    num_classes = train_df[label_col].nunique()
    print(f"✅ Detected {num_classes} classes in dataset")
    
    # Optimized DataLoader settings from AN2DL lectures
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=config.NUM_WORKERS,
        pin_memory=config.PIN_MEMORY,
        pin_memory_device="cuda" if torch.cuda.is_available() else "",
        prefetch_factor=config.PREFETCH_FACTOR if config.NUM_WORKERS > 0 else None,
        drop_last=True  # Drop incomplete batch for training stability
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=config.NUM_WORKERS,
        pin_memory=config.PIN_MEMORY,
        pin_memory_device="cuda" if torch.cuda.is_available() else "",
        prefetch_factor=config.PREFETCH_FACTOR if config.NUM_WORKERS > 0 else None
    )
    
    return train_loader, val_loader, num_classes


class TestDataset(Dataset):
    """Dataset for test images (no labels)"""
    
    def __init__(self, test_dir, transform=None, file_pattern='img_', use_mask=True):
        """
        Args:
            test_dir: Directory containing test images
            transform: Optional transforms to apply
            file_pattern: Pattern to filter files (default: 'img_' to exclude masks)
            use_mask: Whether to apply mask to images if available
        """
        self.test_dir = test_dir
        self.transform = transform
        self.use_mask = use_mask
        
        # Get all image files
        all_files = [f for f in os.listdir(test_dir) 
                    if f.endswith(('.png', '.jpg', '.jpeg'))]
        
        # Filter by pattern (e.g., only img_*.png, not mask_*.png)
        self.image_files = sorted([f for f in all_files if f.startswith(file_pattern)])
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.test_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        
        # Apply mask if available and use_mask=True
        if self.use_mask:
            # Try to find corresponding mask
            mask_name = img_name.replace('img_', 'mask_')
            mask_path = os.path.join(self.test_dir, mask_name)
            
            if os.path.exists(mask_path):
                mask = Image.open(mask_path).convert('L')
                
                # Apply mask - set background to ImageNet mean instead of black
                image_np = np.array(image)
                mask_np = np.array(mask)
                
                # ImageNet mean RGB values (unnormalized)
                imagenet_mean_rgb = np.array([0.485 * 255, 0.456 * 255, 0.406 * 255], dtype=np.uint8)
                
                # Create background with ImageNet mean color
                masked_image = np.where(
                    mask_np[:, :, np.newaxis] > 0,  # Where mask is active
                    image_np,  # Keep original image
                    imagenet_mean_rgb  # Use ImageNet mean for background
                )
                image = Image.fromarray(masked_image.astype(np.uint8))
        
        if self.transform:
            image = self.transform(image)
        
        return image, img_name  # Return filename for submission



# ============================================================
# normalization.py
# ============================================================

"""
Advanced normalization strategies
Implements ADVICE 04/12 - Normalisation Strategies
"""
import torch
import torch.nn as nn


class AdaptiveNormalization(nn.Module):
    """
    Adaptive normalization that switches between BatchNorm and GroupNorm
    based on batch size
    
    For small batches (<16), GroupNorm is more stable than BatchNorm.
    For larger batches (>=16), BatchNorm works well.
    """
    def __init__(self, num_channels, num_groups=32, threshold_batch_size=16):
        super().__init__()
        self.threshold = threshold_batch_size
        self.batch_norm = nn.BatchNorm2d(num_channels)
        self.group_norm = nn.GroupNorm(min(num_groups, num_channels), num_channels)
        self.num_channels = num_channels
    
    def forward(self, x):
        batch_size = x.size(0)
        
        if batch_size < self.threshold:
            # Use GroupNorm for small batches
            return self.group_norm(x)
        else:
            # Use BatchNorm for larger batches
            return self.batch_norm(x)


class LayerNorm2d(nn.Module):
    """
    Layer Normalization for 2D feature maps (images)
    Normalizes across channel, height, and width dimensions
    
    Good for: Variable batch sizes, transformers, any single-sample processing
    """
    def __init__(self, num_channels, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(num_channels))
        self.bias = nn.Parameter(torch.zeros(num_channels))
        self.eps = eps
    
    def forward(self, x):
        # x shape: (B, C, H, W)
        # Normalize over C, H, W for each sample independently
        mean = x.mean(dim=[1, 2, 3], keepdim=True)
        var = x.var(dim=[1, 2, 3], keepdim=True, unbiased=False)
        
        x = (x - mean) / torch.sqrt(var + self.eps)
        
        # Apply learnable affine transform
        x = x * self.weight.view(1, -1, 1, 1) + self.bias.view(1, -1, 1, 1)
        
        return x


class InstanceNorm2d(nn.Module):
    """
    Instance Normalization for 2D feature maps
    Normalizes across spatial dimensions (H, W) for each channel independently
    
    Good for: Style transfer, image generation, when batch statistics are unreliable
    """
    def __init__(self, num_channels, eps=1e-6, affine=True):
        super().__init__()
        self.eps = eps
        self.affine = affine
        
        if affine:
            self.weight = nn.Parameter(torch.ones(num_channels))
            self.bias = nn.Parameter(torch.zeros(num_channels))
    
    def forward(self, x):
        # x shape: (B, C, H, W)
        # Normalize over H, W for each channel independently
        mean = x.mean(dim=[2, 3], keepdim=True)
        var = x.var(dim=[2, 3], keepdim=True, unbiased=False)
        
        x = (x - mean) / torch.sqrt(var + self.eps)
        
        if self.affine:
            x = x * self.weight.view(1, -1, 1, 1) + self.bias.view(1, -1, 1, 1)
        
        return x


def replace_batchnorm_with_groupnorm(model, num_groups=32):
    """
    Replace all BatchNorm layers with GroupNorm in a model
    
    Useful when training with very small batch sizes (common in Colab with limited memory)
    
    Args:
        model: PyTorch model
        num_groups: Number of groups for GroupNorm (default 32)
    
    Returns:
        Modified model
    """
    for name, module in model.named_children():
        if isinstance(module, nn.BatchNorm2d):
            # Replace with GroupNorm
            num_channels = module.num_features
            new_module = nn.GroupNorm(min(num_groups, num_channels), num_channels)
            setattr(model, name, new_module)
        else:
            # Recursively apply to submodules
            replace_batchnorm_with_groupnorm(module, num_groups)
    
    return model


def replace_batchnorm_with_layernorm(model):
    """
    Replace all BatchNorm layers with LayerNorm
    
    Useful for: Transformer models, batch-size-independent training
    
    Args:
        model: PyTorch model
    
    Returns:
        Modified model
    """
    for name, module in model.named_children():
        if isinstance(module, nn.BatchNorm2d):
            num_channels = module.num_features
            new_module = LayerNorm2d(num_channels)
            setattr(model, name, new_module)
        else:
            replace_batchnorm_with_layernorm(module)
    
    return model


def get_normalization_layer(norm_type, num_channels, num_groups=32):
    """
    Factory function to get normalization layer
    
    Args:
        norm_type: 'batch', 'group', 'layer', 'instance', or 'adaptive'
        num_channels: Number of input channels
        num_groups: Number of groups for GroupNorm
    
    Returns:
        Normalization layer
    """
    norm_layers = {
        'batch': lambda: nn.BatchNorm2d(num_channels),
        'group': lambda: nn.GroupNorm(min(num_groups, num_channels), num_channels),
        'layer': lambda: LayerNorm2d(num_channels),
        'instance': lambda: InstanceNorm2d(num_channels),
        'adaptive': lambda: AdaptiveNormalization(num_channels, num_groups)
    }
    
    if norm_type not in norm_layers:
        raise ValueError(f"Normalization type {norm_type} not supported. "
                        f"Choose from: {list(norm_layers.keys())}")
    
    return norm_layers[norm_type]()


def diagnose_batch_size_issues(model, dataloader, device):
    """
    Diagnose if batch size is causing training instability
    
    Compares BatchNorm statistics at different batch sizes
    
    Args:
        model: Model with BatchNorm layers
        dataloader: DataLoader
        device: cuda or cpu
    
    Returns:
        Dictionary with diagnostic information
    """
    model.eval()
    
    bn_layers = []
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            bn_layers.append(module)
    
    if not bn_layers:
        print("⚠️  No BatchNorm layers found in model")
        return {}
    
    print(f"📊 Found {len(bn_layers)} BatchNorm layers")
    print(f"   Current batch size: {dataloader.batch_size}")
    
    # Get one batch and check statistics
    images, _ = next(iter(dataloader))
    images = images.to(device)
    
    with torch.no_grad():
        _ = model(images)
    
    # Check running statistics
    diagnostics = {
        'batch_size': dataloader.batch_size,
        'num_bn_layers': len(bn_layers),
        'running_var_stats': []
    }
    
    for idx, bn in enumerate(bn_layers):
        if bn.running_var is not None:
            var_mean = bn.running_var.mean().item()
            var_std = bn.running_var.std().item()
            diagnostics['running_var_stats'].append({
                'layer': idx,
                'var_mean': var_mean,
                'var_std': var_std,
                'stable': var_std < 1.0  # Heuristic for stability
            })
    
    # Recommendation
    unstable_layers = sum(1 for s in diagnostics['running_var_stats'] if not s['stable'])
    
    if unstable_layers > len(bn_layers) * 0.3:
        print(f"\n⚠️  WARNING: {unstable_layers}/{len(bn_layers)} BatchNorm layers may be unstable")
        print(f"   Consider:")
        print(f"   1. Increasing batch size (current: {dataloader.batch_size})")
        print(f"   2. Using GroupNorm instead of BatchNorm")
        print(f"   3. Using LayerNorm for batch-size independence")
    else:
        print(f"\n✅ BatchNorm statistics look stable")
    
    return diagnostics



# ============================================================
# models.py
# ============================================================

"""
Model architectures with attention mechanisms - config-driven
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision.models import (
    ResNet18_Weights, ResNet34_Weights, ResNet50_Weights, ResNet101_Weights, ResNet152_Weights,
    EfficientNet_B0_Weights, EfficientNet_V2_S_Weights, EfficientNet_V2_M_Weights,
    MobileNet_V3_Large_Weights, MobileNet_V3_Small_Weights,
    VGG16_Weights, DenseNet121_Weights,
    ConvNeXt_Tiny_Weights, ConvNeXt_Small_Weights
)


class MaskedSpatialAttention(nn.Module):
    """Spatial attention mechanism for feature maps"""
    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels // 8, kernel_size=1)
        self.conv2 = nn.Conv2d(in_channels // 8, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # Generate attention map
        attention = self.conv1(x)
        attention = F.relu(attention)
        attention = self.conv2(attention)
        attention = self.sigmoid(attention)
        
        # Apply attention
        return x * attention, attention


class ChannelAttention(nn.Module):
    """Channel attention mechanism (Squeeze-and-Excitation)"""
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class AttentionWrapper(nn.Module):
    """Wraps model with attention mechanisms
    
    NOTE: Attention mechanisms require 4D feature maps (B, C, H, W).
    For models that internally pool features, attention is disabled automatically.
    """
    def __init__(self, backbone, num_classes=8, dropout=0.5, 
                 use_spatial_attention=True, use_channel_attention=True):
        super().__init__()
        self.backbone = backbone
        self.use_spatial_attention = use_spatial_attention
        self.use_channel_attention = use_channel_attention
        
        # Get feature dimension and remove original classifier
        if hasattr(backbone, 'fc'):
            # ResNet-style: has avgpool and fc
            self.feature_dim = backbone.fc.in_features
            backbone.fc = nn.Identity()
            # Remove avgpool to get feature maps for attention
            if hasattr(backbone, 'avgpool'):
                backbone.avgpool = nn.Identity()
                self.needs_pooling = True
            else:
                self.needs_pooling = False
        elif hasattr(backbone, 'classifier'):
            # EfficientNet/MobileNet-style: has classifier
            if isinstance(backbone.classifier, nn.Sequential):
                # Find the last Linear layer in the sequential
                for layer in reversed(backbone.classifier):
                    if isinstance(layer, nn.Linear):
                        self.feature_dim = layer.in_features
                        break
            else:
                self.feature_dim = backbone.classifier.in_features
            backbone.classifier = nn.Identity()
            
            # For EfficientNet/MobileNet, pooling happens inside features module
            # We can't easily remove it, so we'll detect 2D output and handle it
            self.needs_pooling = False
        
        # Attention modules operate on feature maps (will be disabled if output is 2D)
        if use_spatial_attention:
            self.spatial_attention = MaskedSpatialAttention(self.feature_dim)
        if use_channel_attention:
            self.channel_attention = ChannelAttention(self.feature_dim)
        
        # Adaptive classifier - dimension will be adjusted on first forward pass if needed
        self.classifier = None
        self.num_classes = num_classes
        self.dropout = dropout
        
        self.last_attention_map = None
        
    def forward(self, x):
        features = self.backbone(x)
        
        # Lazy initialization of classifier based on actual feature shape
        if self.classifier is None:
            if len(features.shape) == 4:
                # 4D feature maps - attention can be applied
                actual_dim = features.size(1)  # Channel dimension
            else:
                # 2D features - already pooled internally
                actual_dim = features.size(1)
                # Disable attention for 2D features
                self.use_spatial_attention = False
                self.use_channel_attention = False
            
            self.classifier = nn.Sequential(
                nn.BatchNorm1d(actual_dim),
                nn.Dropout(self.dropout),
                nn.Linear(actual_dim, self.num_classes)
            ).to(features.device)
        
        # Handle case where backbone outputs 2D (some models pool internally)
        if len(features.shape) == 2:
            # Features are already pooled, skip attention
            return self.classifier(features)
        
        # Now we have 4D feature maps (batch, channels, height, width)
        if self.use_channel_attention:
            features = self.channel_attention(features)
        
        if self.use_spatial_attention:
            features, attention_map = self.spatial_attention(features)
            self.last_attention_map = attention_map.detach()
        
        # Global pooling
        features = F.adaptive_avg_pool2d(features, (1, 1))
        features = features.view(features.size(0), -1)
        
        return self.classifier(features)
    
    def get_attention_map(self):
        """Get last attention map for visualization"""
        return self.last_attention_map


def get_model(config):
    """Build model from config
    
    Supported models:
        resnet18/34/50/101/152, efficientnet_b0/v2_s/v2_m, mobilenet_v3_large/small,
        vgg16, densenet121, convnext_tiny/small
    """
    model_name = config.MODEL_NAME.lower()
    num_classes = config.NUM_CLASSES
    pretrained = getattr(config, 'USE_PRETRAINED')
    use_attention = getattr(config, 'USE_ATTENTION')
    dropout = getattr(config, 'DROPOUT')
    norm_type = getattr(config, 'NORM_TYPE')
    batch_size = getattr(config, 'BATCH_SIZE')
    
    # Model registry
    models_registry = {
        'resnet18': (models.resnet18, ResNet18_Weights.IMAGENET1K_V1, 'fc'),
        'resnet34': (models.resnet34, ResNet34_Weights.IMAGENET1K_V1, 'fc'),
        'resnet50': (models.resnet50, ResNet50_Weights.IMAGENET1K_V2, 'fc'),
        'resnet101': (models.resnet101, ResNet101_Weights.IMAGENET1K_V2, 'fc'),
        'resnet152': (models.resnet152, ResNet152_Weights.IMAGENET1K_V2, 'fc'),
        'efficientnet_b0': (models.efficientnet_b0, EfficientNet_B0_Weights.IMAGENET1K_V1, 'classifier'),
        'efficientnet_v2_s': (models.efficientnet_v2_s, EfficientNet_V2_S_Weights.IMAGENET1K_V1, 'classifier'),
        'efficientnet_v2_m': (models.efficientnet_v2_m, EfficientNet_V2_M_Weights.IMAGENET1K_V1, 'classifier'),
        'mobilenet_v3_large': (models.mobilenet_v3_large, MobileNet_V3_Large_Weights.IMAGENET1K_V2, 'classifier'),
        'mobilenet_v3_small': (models.mobilenet_v3_small, MobileNet_V3_Small_Weights.IMAGENET1K_V1, 'classifier'),
        'vgg16': (models.vgg16_bn, VGG16_Weights.IMAGENET1K_V1, 'classifier'),
        'densenet121': (models.densenet121, DenseNet121_Weights.IMAGENET1K_V1, 'classifier'),
        'convnext_tiny': (models.convnext_tiny, ConvNeXt_Tiny_Weights.IMAGENET1K_V1, 'classifier'),
        'convnext_small': (models.convnext_small, ConvNeXt_Small_Weights.IMAGENET1K_V1, 'classifier'),
    }
    
    if model_name not in models_registry:
        raise ValueError(f"Unknown model: {model_name}. Choose from: {list(models_registry.keys())}")
    
    model_fn, weights, classifier_name = models_registry[model_name]
    model = model_fn(weights=weights if pretrained else None)
    
    # Replace classifier
    if classifier_name == 'fc':
        num_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.BatchNorm1d(num_features),
            nn.Dropout(dropout),
            nn.Linear(num_features, num_classes)
        )
    elif classifier_name == 'classifier':
        if isinstance(model.classifier, nn.Sequential):
            # Find the last Linear layer in the sequential
            for layer in reversed(model.classifier):
                if isinstance(layer, nn.Linear):
                    num_features = layer.in_features
                    break
            model.classifier[-1] = nn.Linear(num_features, num_classes)
            # Insert batch norm and dropout before final layer
            model.classifier = nn.Sequential(
                *list(model.classifier[:-1]),
                nn.BatchNorm1d(num_features),
                nn.Dropout(dropout),
                model.classifier[-1]
            )
        else:
            num_features = model.classifier.in_features
            model.classifier = nn.Sequential(
                nn.BatchNorm1d(num_features),
                nn.Dropout(dropout),
                nn.Linear(num_features, num_classes)
            )
    
    # Wrap with attention if requested
    if use_attention:
        model = AttentionWrapper(model, num_classes=num_classes, dropout=dropout)
    
    # Apply normalization strategy - ADVICE 04/12
    # "Do not force the ocean's rules upon a cup of water"
    if norm_type == 'group' and batch_size < 16:
        # replace_batchnorm_with_groupnorm imported at module level
        model = replace_batchnorm_with_groupnorm(model, num_groups=32)
        print(f"   ✅ Applied GroupNorm (batch_size={batch_size} < 16)")
    elif norm_type == 'layer':
        # replace_batchnorm_with_layernorm imported at module level
        model = replace_batchnorm_with_layernorm(model)
        print(f"   ✅ Applied LayerNorm2d")
    
    # Freeze backbone if transfer learning
    if getattr(config, 'FREEZE_BACKBONE'):
        freeze_backbone(model, freeze=True)
    
    params = count_parameters(model)
    print(f"✅ Model: {model_name} | Params: {params:,} | Pretrained: {pretrained}")
    
    return model


def count_parameters(model):
    """Count trainable parameters"""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable


def freeze_backbone(model, freeze=True):
    """Freeze/unfreeze backbone for transfer learning"""
    if isinstance(model, AttentionWrapper):
        target = model.backbone
    else:
        target = model
    
    # Freeze all backbone parameters
    for param in target.parameters():
        param.requires_grad = not freeze
    
    # Keep classifier trainable
    if isinstance(model, AttentionWrapper):
        for param in model.classifier.parameters():
            param.requires_grad = True
        if model.use_spatial_attention:
            for param in model.spatial_attention.parameters():
                param.requires_grad = True
        if model.use_channel_attention:
            for param in model.channel_attention.parameters():
                param.requires_grad = True
    else:
        if hasattr(target, 'fc'):
            for param in target.fc.parameters():
                param.requires_grad = True
        elif hasattr(target, 'classifier'):
            for param in target.classifier.parameters():
                param.requires_grad = True
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    
    print(f"{'🔒 Frozen' if freeze else '🔓 Unfrozen'} backbone: {trainable:,} / {total:,} params trainable")
    
    return model



# ============================================================
# advanced_optimizers.py
# ============================================================

"""
Advanced optimizers: Lion, Ranger, and others
Implements ADVICE 07/12 - Modern Optimizers
"""
import torch
from torch.optim.optimizer import Optimizer
import math


class Lion(Optimizer):
    """
    Lion optimizer (EvoLved Sign Momentum)
    Paper: https://arxiv.org/abs/2302.06675
    
    Lion uses sign of gradients and is more memory efficient than Adam.
    Typically use learning_rate = lr_adam / 3, weight_decay = wd_adam * 3
    """
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError(f"Invalid beta parameter at index 0: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError(f"Invalid beta parameter at index 1: {betas[1]}")
        
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)
    
    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                
                # Perform stepweight decay
                p.data.mul_(1 - group['lr'] * group['weight_decay'])
                
                grad = p.grad
                state = self.state[p]
                
                # State initialization
                if len(state) == 0:
                    state['exp_avg'] = torch.zeros_like(p)
                
                exp_avg = state['exp_avg']
                beta1, beta2 = group['betas']
                
                # Weight update
                update = exp_avg * beta1 + grad * (1 - beta1)
                p.add_(torch.sign(update), alpha=-group['lr'])
                
                # Momentum update
                exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)
        
        return loss


class Ranger(Optimizer):
    """
    Ranger optimizer = RAdam + Lookahead
    Combines rectified adaptive learning rates with lookahead mechanism
    """
    def __init__(self, params, lr=1e-3, alpha=0.5, k=6, betas=(0.9, 0.999), 
                 eps=1e-8, weight_decay=0):
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        if not 0.0 <= eps:
            raise ValueError(f"Invalid epsilon value: {eps}")
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError(f"Invalid beta parameter at index 0: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError(f"Invalid beta parameter at index 1: {betas[1]}")
        if not 0.0 <= alpha <= 1.0:
            raise ValueError(f"Invalid alpha parameter: {alpha}")
        
        defaults = dict(lr=lr, alpha=alpha, k=k, betas=betas, eps=eps, 
                       weight_decay=weight_decay)
        super().__init__(params, defaults)
        
        # Lookahead slow weights
        self.buffer = [[p.clone().detach() for p in group['params']] 
                       for group in self.param_groups]
    
    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                
                grad = p.grad
                if grad.is_sparse:
                    raise RuntimeError('Ranger does not support sparse gradients')
                
                state = self.state[p]
                
                # State initialization
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                beta1, beta2 = group['betas']
                state['step'] += 1
                
                # Decay the first and second moment running average coefficient
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                
                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']
                
                # RAdam variance rectification
                rho_inf = 2 / (1 - beta2) - 1
                rho = rho_inf - 2 * state['step'] * (beta2 ** state['step']) / bias_correction2
                
                if rho > 4:
                    # Adaptive learning rate
                    rect = math.sqrt(
                        (rho - 4) * (rho - 2) * rho_inf / 
                        ((rho_inf - 4) * (rho_inf - 2) * rho)
                    )
                    denom = exp_avg_sq.sqrt().add_(group['eps'])
                    step_size = group['lr'] * rect / bias_correction1
                    p.addcdiv_(exp_avg, denom, value=-step_size)
                else:
                    # Use SGD-like step when variance not reliable
                    step_size = group['lr'] / bias_correction1
                    p.add_(exp_avg, alpha=-step_size)
                
                # Weight decay
                if group['weight_decay'] != 0:
                    p.add_(p, alpha=-group['lr'] * group['weight_decay'])
        
        # Lookahead step
        for group, buffer_group in zip(self.param_groups, self.buffer):
            k = group['k']
            alpha = group['alpha']
            
            if self.state[group['params'][0]]['step'] % k == 0:
                for p, slow_p in zip(group['params'], buffer_group):
                    slow_p.add_(p - slow_p, alpha=alpha)
                    p.copy_(slow_p)
        
        return loss


def get_optimizer(name, model_parameters, lr, weight_decay):
    """
    Factory function to get optimizer by name
    
    Args:
        name: 'adamw', 'lion', 'ranger'
        model_parameters: model.parameters()
        lr: learning rate
        weight_decay: weight decay factor
    
    Returns:
        Optimizer instance
    """
    optimizers = {
        'adamw': lambda: torch.optim.AdamW(
            model_parameters,
            lr=lr,
            weight_decay=weight_decay
        ),
        'lion': lambda: Lion(
            model_parameters,
            lr=lr / 3,  # Lion typically uses lower LR
            weight_decay=weight_decay * 3  # But higher weight decay
        ),
        'ranger': lambda: Ranger(
            model_parameters,
            lr=lr,
            weight_decay=weight_decay
        )
    }
    
    if name not in optimizers:
        raise ValueError(f"Optimizer {name} not supported. Choose from {list(optimizers.keys())}")
    
    return optimizers[name]()



# ============================================================
# mixup.py
# ============================================================

"""
Advanced augmentation techniques: CutMix, MixUp
Based on PERFORMANCE_ROADMAP recommendations
"""
import torch
import numpy as np


class CutMix:
    """CutMix augmentation for training
    
    Cuts a random region from one image and pastes it into another,
    with labels mixed proportionally to the area.
    
    Paper: CutMix: Regularization Strategy to Train Strong Classifiers
    https://arxiv.org/abs/1905.04899
    """
    def __init__(self, alpha=1.0, prob=0.5):
        """
        Args:
            alpha: Beta distribution parameter (higher = more mixing)
            prob: Probability of applying CutMix
        """
        self.alpha = alpha
        self.prob = prob
    
    def __call__(self, batch_images, batch_labels):
        """Apply CutMix to a batch
        
        Args:
            batch_images: Tensor of shape (B, C, H, W)
            batch_labels: Tensor of shape (B,) with class indices
            
        Returns:
            mixed_images: CutMix augmented images
            labels_a: Original labels
            labels_b: Mixed labels
            lam: Mixing coefficient
        """
        if np.random.rand() > self.prob:
            return batch_images, batch_labels, batch_labels, 1.0
        
        batch_size = batch_images.size(0)
        indices = torch.randperm(batch_size).to(batch_images.device)
        
        # Sample lambda from beta distribution
        lam = np.random.beta(self.alpha, self.alpha)
        
        # Generate random box
        _, _, H, W = batch_images.shape
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)
        
        # Random center point
        cx = np.random.randint(W)
        cy = np.random.randint(H)
        
        # Box coordinates
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)
        
        # Cut and paste
        batch_images[:, :, bby1:bby2, bbx1:bbx2] = batch_images[indices, :, bby1:bby2, bbx1:bbx2]
        
        # Adjust lambda based on actual box size
        lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (W * H))
        
        return batch_images, batch_labels, batch_labels[indices], lam


class MixUp:
    """MixUp augmentation for training
    
    Linearly interpolates two images and their labels.
    
    Paper: mixup: Beyond Empirical Risk Minimization
    https://arxiv.org/abs/1710.09412
    """
    def __init__(self, alpha=0.2, prob=0.5):
        """
        Args:
            alpha: Beta distribution parameter (lower = less mixing for medical images)
            prob: Probability of applying MixUp
        """
        self.alpha = alpha
        self.prob = prob
    
    def __call__(self, batch_images, batch_labels):
        """Apply MixUp to a batch
        
        Args:
            batch_images: Tensor of shape (B, C, H, W)
            batch_labels: Tensor of shape (B,) with class indices
            
        Returns:
            mixed_images: MixUp augmented images
            labels_a: Original labels
            labels_b: Mixed labels
            lam: Mixing coefficient
        """
        if np.random.rand() > self.prob:
            return batch_images, batch_labels, batch_labels, 1.0
        
        batch_size = batch_images.size(0)
        indices = torch.randperm(batch_size).to(batch_images.device)
        
        # Sample lambda from beta distribution
        lam = np.random.beta(self.alpha, self.alpha)
        
        # Mix images
        mixed_images = lam * batch_images + (1 - lam) * batch_images[indices]
        
        return mixed_images, batch_labels, batch_labels[indices], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Loss function for MixUp/CutMix
    
    Args:
        criterion: Loss function (e.g., CrossEntropyLoss)
        pred: Model predictions
        y_a: Original labels
        y_b: Mixed labels
        lam: Mixing coefficient
        
    Returns:
        Mixed loss
    """
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)



# ============================================================
# trainer.py
# ============================================================

"""
Streamlined trainer with TensorBoard, mixed precision, and checkpointing
"""
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.tensorboard import SummaryWriter


class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance"""
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss


class Trainer:
    """Production-ready trainer with all features integrated"""
    
    def __init__(self, model, train_loader, val_loader, config, device):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        self.device = device
        
        # Loss function
        label_smoothing = getattr(config, 'LABEL_SMOOTHING')
        use_focal_loss = getattr(config, 'USE_FOCAL_LOSS')
        use_weighted_loss = getattr(config, 'USE_WEIGHTED_LOSS')
        
        if use_focal_loss:
            focal_alpha = getattr(config, 'FOCAL_ALPHA')
            focal_gamma = getattr(config, 'FOCAL_GAMMA')
            self.criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)
            print(f"📊 Using Focal Loss (alpha={focal_alpha}, gamma={focal_gamma})")
        elif use_weighted_loss:
            # Class weights: inverse frequency normalized
            # Triple neg: 156, Luminal A: 414, Luminal B: 445, HER2+: 397
            class_weights = getattr(config, 'CLASS_WEIGHTS')
            weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
            self.criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=label_smoothing)
            print(f"⚖️  Using Weighted Loss: {class_weights}")
        else:
            self.criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        
        # Optimizer
        optimizer_name = config.OPTIMIZER.lower()
        if optimizer_name == 'adam':
            self.optimizer = torch.optim.Adam(
                model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
        elif optimizer_name == 'sgd':
            self.optimizer = torch.optim.SGD(
                model.parameters(), lr=config.LEARNING_RATE, 
                weight_decay=config.WEIGHT_DECAY, momentum=0.9, nesterov=True)
        elif optimizer_name == 'lion':
            # Lion imported at module level
            self.optimizer = Lion(
                model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
        elif optimizer_name == 'ranger':
            # Ranger imported at module level
            self.optimizer = Ranger(
                model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
        else:  # adamw (default)
            self.optimizer = torch.optim.AdamW(
                model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
        
        # Scheduler
        self.scheduler = None
        if getattr(config, 'USE_SCHEDULER'):
            scheduler_type = getattr(config, 'SCHEDULER').lower()
            if scheduler_type == 'plateau':
                self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    self.optimizer, mode='max', 
                    factor=getattr(config, 'FACTOR'),
                    patience=getattr(config, 'PATIENCE_SCHEDULER'),
                    min_lr=1e-6)
            elif scheduler_type == 'cosine':
                self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    self.optimizer, T_max=config.NUM_EPOCHS, eta_min=1e-6)
            elif scheduler_type == 'step':
                self.scheduler = torch.optim.lr_scheduler.StepLR(
                    self.optimizer, step_size=15, gamma=0.1)
        
        # Mixed precision
        self.use_amp = getattr(config, 'USE_MIXED_PRECISION') and device.type == 'cuda'
        self.scaler = torch.amp.GradScaler(enabled=self.use_amp)
        self.gradient_clip = getattr(config, 'GRADIENT_CLIP')
        
        # TensorBoard
        self.writer = None
        if getattr(config, 'USE_TENSORBOARD'):
            log_dir = f"{config.TENSORBOARD_DIR}/{time.strftime('%Y%m%d_%H%M%S')}"
            self.writer = SummaryWriter(log_dir)
            print(f"✅ TensorBoard: {log_dir}")
        
        # Training state
        self.current_epoch = 0
        self.best_metric = 0.0
        self.history = []
        self.patience_counter = 0
        
        # CutMix/MixUp augmentation
        self.use_cutmix = getattr(config, 'USE_CUTMIX')
        self.use_mixup = getattr(config, 'USE_MIXUP')
        if self.use_cutmix or self.use_mixup:
            # CutMix/MixUp imported at module level
            self.mixup_fn = None
            if self.use_cutmix:
                cutmix_alpha = getattr(config, 'CUTMIX_ALPHA')
                cutmix_prob = getattr(config, 'CUTMIX_PROB')
                self.mixup_fn = CutMix(alpha=cutmix_alpha, prob=cutmix_prob)
                print(f"✂️  Using CutMix (alpha={cutmix_alpha}, prob={cutmix_prob})")
            elif self.use_mixup:
                mixup_alpha = getattr(config, 'MIXUP_ALPHA')
                mixup_prob = getattr(config, 'MIXUP_PROB')
                self.mixup_fn = MixUp(alpha=mixup_alpha, prob=mixup_prob)
                print(f"🔀 Using MixUp (alpha={mixup_alpha}, prob={mixup_prob})")
            self.mixup_criterion = mixup_criterion
        else:
            self.mixup_fn = None
        
        print(f"✅ Trainer ready - {optimizer_name.upper()}, AMP={self.use_amp}")
    
    def train_epoch(self):
        """Train one epoch"""
        self.model.train()
        running_loss = 0.0
        all_preds, all_labels = [], []
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {self.current_epoch+1}")
        
        for batch_idx, (images, labels) in enumerate(pbar):
            images, labels = images.to(self.device), labels.to(self.device)
            
            # Apply CutMix/MixUp if enabled
            if self.mixup_fn is not None:
                images, labels_a, labels_b, lam = self.mixup_fn(images, labels)
                use_mixup = True
            else:
                labels_a = labels
                labels_b = None
                lam = 1.0
                use_mixup = False
            
            self.optimizer.zero_grad(set_to_none=True)
            
            # Forward with AMP
            with torch.amp.autocast(device_type=self.device.type, enabled=self.use_amp):
                outputs = self.model(images)
                if use_mixup:
                    loss = self.mixup_criterion(self.criterion, outputs, labels_a, labels_b, lam)
                else:
                    loss = self.criterion(outputs, labels)
            
            # Backward
            if self.use_amp:
                self.scaler.scale(loss).backward()
                if self.gradient_clip:
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.gradient_clip)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                loss.backward()
                if self.gradient_clip:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.gradient_clip)
                self.optimizer.step()
            
            # Metrics (use original labels for tracking)
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_a.cpu().numpy())
            
            # TensorBoard batch logging
            if self.writer and batch_idx % getattr(self.config, 'LOG_INTERVAL') == 0:
                step = self.current_epoch * len(self.train_loader) + batch_idx
                self.writer.add_scalar('Batch/Loss', loss.item(), step)
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        epoch_loss = running_loss / len(self.train_loader.dataset)
        epoch_acc = accuracy_score(all_labels, all_preds)
        epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
        
        return epoch_loss, epoch_acc, epoch_f1
    
    def validate(self):
        """Validate model"""
        self.model.eval()
        running_loss = 0.0
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            for images, labels in tqdm(self.val_loader, desc="Validating", leave=False):
                images, labels = images.to(self.device), labels.to(self.device)
                
                with torch.amp.autocast(device_type=self.device.type, enabled=self.use_amp):
                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)
                
                running_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        val_loss = running_loss / len(self.val_loader.dataset)
        val_acc = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds, average='weighted')
        
        return val_loss, val_acc, val_f1
    
    def save_checkpoint(self, is_best=False):
        """Save checkpoint"""
        checkpoint = {
            'epoch': self.current_epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'best_metric': self.best_metric,
            'history': self.history
        }
        if self.scheduler:
            checkpoint['scheduler_state_dict'] = self.scheduler.state_dict()
        if self.use_amp:
            checkpoint['scaler_state_dict'] = self.scaler.state_dict()
        
        # Save latest
        latest_path = os.path.join(self.config.CHECKPOINT_DIR, 'latest.pt')
        torch.save(checkpoint, latest_path)
        
        # Save best
        if is_best:
            exp_name = getattr(self.config, 'EXPERIMENT_NAME')
            best_path = os.path.join(self.config.MODELS_DIR, f'{exp_name}_best.pt')
            torch.save(self.model.state_dict(), best_path)
            print(f"💾 Best: {self.best_metric:.4f} -> {best_path}")
    
    def freeze_backbone(self):
        """Freeze backbone layers for progressive training"""
        frozen_count = 0
        for name, param in self.model.named_parameters():
            # Freeze all except classifier/fc layer
            if 'classifier' not in name and 'fc' not in name:
                param.requires_grad = False
                frozen_count += 1
        print(f"❄️  Froze {frozen_count} backbone parameters")
    
    def unfreeze_backbone(self):
        """Unfreeze all layers for fine-tuning"""
        unfrozen_count = 0
        for param in self.model.parameters():
            if not param.requires_grad:
                param.requires_grad = True
                unfrozen_count += 1
        print(f"🔥 Unfroze {unfrozen_count} parameters for fine-tuning")
    
    def load_checkpoint(self, path=None):
        """Load checkpoint"""
        if path is None:
            path = os.path.join(self.config.CHECKPOINT_DIR, 'latest.pt')
        
        if not os.path.exists(path):
            return False
        
        checkpoint = torch.load(path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        if self.scheduler and 'scheduler_state_dict' in checkpoint:
            self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        if self.use_amp and 'scaler_state_dict' in checkpoint:
            self.scaler.load_state_dict(checkpoint['scaler_state_dict'])
        self.current_epoch = checkpoint['epoch'] + 1
        self.best_metric = checkpoint['best_metric']
        self.history = checkpoint['history']
        
        print(f"✅ Resumed from epoch {self.current_epoch}")
        return True
    
    def train(self, resume=False):
        """Main training loop"""
        if resume:
            self.load_checkpoint()
        
        print(f"\n🚀 Training: {self.config.NUM_EPOCHS} epochs")
        print(f"   Device: {self.device} | Batch: {self.config.BATCH_SIZE} | LR: {self.config.LEARNING_RATE}")
        
        start_time = time.time()
        
        for epoch in range(self.current_epoch, self.config.NUM_EPOCHS):
            self.current_epoch = epoch
            
            # Train & validate
            train_loss, train_acc, train_f1 = self.train_epoch()
            val_loss, val_acc, val_f1 = self.validate()
            
            # Update scheduler
            if self.scheduler:
                if isinstance(self.scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    self.scheduler.step(val_f1)
                else:
                    self.scheduler.step()
            
            # Track history
            self.history.append({
                'epoch': epoch + 1,
                'train_loss': train_loss, 'train_acc': train_acc, 'train_f1': train_f1,
                'val_loss': val_loss, 'val_acc': val_acc, 'val_f1': val_f1,
                'lr': self.optimizer.param_groups[0]['lr']
            })
            
            # TensorBoard
            if self.writer:
                self.writer.add_scalars('Loss', {'train': train_loss, 'val': val_loss}, epoch)
                self.writer.add_scalars('Accuracy', {'train': train_acc, 'val': val_acc}, epoch)
                self.writer.add_scalars('F1', {'train': train_f1, 'val': val_f1}, epoch)
                self.writer.add_scalar('LR', self.optimizer.param_groups[0]['lr'], epoch)
                
                # Log model weights
                if getattr(self.config, 'LOG_HISTOGRAMS'):
                    for name, param in self.model.named_parameters():
                        if param.requires_grad:
                            self.writer.add_histogram(f'Weights/{name}', param.data, epoch)
                            if param.grad is not None:
                                self.writer.add_histogram(f'Gradients/{name}', param.grad, epoch)
            
            # Print
            print(f"\n[{epoch+1}/{self.config.NUM_EPOCHS}]")
            print(f"  Train: Loss={train_loss:.4f} Acc={train_acc:.4f} F1={train_f1:.4f}")
            print(f"  Val:   Loss={val_loss:.4f} Acc={val_acc:.4f} F1={val_f1:.4f}")
            
            # Check improvement
            monitor_metric = getattr(self.config, 'MONITOR_METRIC')
            current_metric = val_f1 if 'f1' in monitor_metric else (val_acc if 'acc' in monitor_metric else -val_loss)
            
            is_best = current_metric > self.best_metric
            if is_best:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
            
            # Save
            if getattr(self.config, 'SAVE_BEST_ONLY'):
                if is_best:
                    self.save_checkpoint(is_best=True)
            else:
                self.save_checkpoint(is_best=is_best)
                if (epoch + 1) % self.config.CHECKPOINT_INTERVAL == 0:
                    epoch_path = os.path.join(self.config.CHECKPOINT_DIR, f'epoch_{epoch+1}.pt')
                    torch.save(self.model.state_dict(), epoch_path)
            
            # Early stopping
            if self.patience_counter >= self.config.PATIENCE:
                print(f"\n⏹️  Early stop ({self.config.PATIENCE} epochs no improvement)")
                break
        
        elapsed = (time.time() - start_time) / 60
        print(f"\n✅ Done in {elapsed:.1f}min | Best: {self.best_metric:.4f}")
        
        if self.writer:
            self.writer.close()
        
        return self.history
    
    def get_history_df(self):
        """Get training history as pandas DataFrame"""
        return pd.DataFrame(self.history)



# ============================================================
# evaluation.py
# ============================================================

"""
Evaluation and inference utilities
"""
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)


def predict_with_tta(model, image, device, num_augmentations=5, img_size=256):
    """Test-Time Augmentation for robust predictions
    
    Averages predictions over multiple augmented versions of the same image.
    This improves robustness and typically adds +1-2% accuracy.
    
    Args:
        model: Trained model
        image: Single image tensor (C, H, W) or (B, C, H, W)
        device: Device to run on
        num_augmentations: Number of augmented predictions to average
        img_size: Image size for transforms
        
    Returns:
        averaged_logits: Mean logits across all augmentations
        averaged_probs: Mean probabilities across all augmentations
    """
    model.eval()
    
    # Handle batch dimension
    if image.dim() == 3:
        image = image.unsqueeze(0)
    
    # TTA transforms (geometric only, no color changes)
    tta_transforms = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=10)
    ])
    
    predictions = []
    
    with torch.no_grad():
        # Original prediction
        img_device = image.to(device)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            pred = model(img_device)
        predictions.append(pred)
        
        # Augmented predictions
        for _ in range(num_augmentations - 1):
            aug_image = tta_transforms(image).to(device)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                pred = model(aug_image)
            predictions.append(pred)
    
    # Average predictions
    avg_logits = torch.stack(predictions).mean(dim=0)
    avg_probs = F.softmax(avg_logits, dim=1)
    
    return avg_logits, avg_probs


def evaluate_model(model, data_loader, device, class_names=None):
    """Comprehensive model evaluation
    
    Args:
        model: Trained model
        data_loader: Test/validation dataloader
        device: Device to run on
        class_names: List of class names for reporting
    
    Returns:
        Dictionary with all metrics and predictions
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                outputs = model(images)
            
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    # Calculate metrics
    results = {
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, average='weighted'),
        'recall': recall_score(all_labels, all_preds, average='weighted'),
        'f1': f1_score(all_labels, all_preds, average='weighted'),
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs,
        'confusion_matrix': confusion_matrix(all_labels, all_preds)
    }
    
    # Print report
    print("\n📊 Evaluation Results:")
    print(f"   Accuracy:  {results['accuracy']:.4f}")
    print(f"   Precision: {results['precision']:.4f}")
    print(f"   Recall:    {results['recall']:.4f}")
    print(f"   F1 Score:  {results['f1']:.4f}")
    
    if class_names:
        print("\n📋 Classification Report:")
        print(classification_report(all_labels, all_preds, target_names=class_names))
    
    return results


def plot_confusion_matrix(cm, class_names=None, figsize=(10, 8), save_path=None):
    """Plot confusion matrix heatmap
    
    Args:
        cm: Confusion matrix from sklearn
        class_names: List of class names
        figsize: Figure size
        save_path: Optional path to save the plot
    """
    plt.figure(figsize=figsize)
    
    # Normalize confusion matrix
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Plot heatmap
    sns.heatmap(
        cm_normalized,
        annot=True,
        fmt='.2%',
        cmap='Blues',
        xticklabels=class_names or range(cm.shape[0]),
        yticklabels=class_names or range(cm.shape[0]),
        cbar_kws={'label': 'Normalized Count'}
    )
    
    plt.title('Confusion Matrix', fontsize=16, pad=20)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    
    plt.show()


def plot_training_history(history_df, save_path=None):
    """Plot training history curves
    
    Args:
        history_df: DataFrame with training history
        save_path: Optional path to save the plot
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Loss
    axes[0].plot(history_df['epoch'], history_df['train_loss'], label='Train', marker='o')
    axes[0].plot(history_df['epoch'], history_df['val_loss'], label='Val', marker='s')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(history_df['epoch'], history_df['train_acc'], label='Train', marker='o')
    axes[1].plot(history_df['epoch'], history_df['val_acc'], label='Val', marker='s')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # F1 Score
    axes[2].plot(history_df['epoch'], history_df['train_f1'], label='Train', marker='o')
    axes[2].plot(history_df['epoch'], history_df['val_f1'], label='Val', marker='s')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('F1 Score')
    axes[2].set_title('F1 Score')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    
    plt.show()


def predict_single_image(model, image_path, transform, device, class_names=None):
    """Predict on a single image
    
    Args:
        model: Trained model
        image_path: Path to image file
        transform: Transforms to apply
        device: Device to run on
        class_names: List of class names
    
    Returns:
        Dictionary with prediction, probability, and class name
    """
    from PIL import Image
    
    model.eval()
    
    # Load and transform image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            output = model(image_tensor)
        
        probs = torch.softmax(output, dim=1)
        pred_idx = probs.argmax(dim=1).item()
        confidence = probs[0, pred_idx].item()
    
    result = {
        'prediction': pred_idx,
        'confidence': confidence,
        'class_name': class_names[pred_idx] if class_names else str(pred_idx),
        'all_probabilities': probs[0].cpu().numpy()
    }
    
    return result


def create_submission(model, test_dir, config, device, output_path='submission.csv', 
                     class_to_label=None, use_mask=True, use_tta=False, tta_num_aug=5):
    """Create submission file for test set
    
    Args:
        model: Trained model
        test_dir: Directory containing test images
        config: Config object with IMG_SIZE and normalization settings
        device: Device to run on
        output_path: Path to save submission CSV
        class_to_label: Optional dict mapping class indices to label names
        use_mask: Whether to apply masks if available (default: True)
        use_tta: Whether to use Test-Time Augmentation (default: False, +1-2% accuracy)
        tta_num_aug: Number of TTA augmentations if use_tta=True
    
    Returns:
        submission_df: DataFrame with predictions
    """
    import pandas as pd
    from tqdm import tqdm
    from torch.utils.data import DataLoader
    # TestDataset and get_transforms imported at module level
    
    # Create test dataset with masks if available
    _, test_transform = get_transforms(
        img_size=config.IMG_SIZE,
        augment=False,
        use_imagenet_norm=config.USE_IMAGENET_NORM
    )
    
    test_dataset = TestDataset(
        test_dir=test_dir,
        transform=test_transform,
        file_pattern='img_',
        use_mask=use_mask
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=config.NUM_WORKERS,
        pin_memory=config.PIN_MEMORY
    )
    
    model.eval()
    all_preds = []
    all_filenames = []
    
    if use_tta:
        print(f"🔄 Using Test-Time Augmentation ({tta_num_aug} augmentations per image)")
        # TTA: Process images individually for augmentation
        for images, filenames in tqdm(test_loader, desc="Generating predictions with TTA"):
            batch_preds = []
            for img in images:
                # Apply TTA to each image
                _, avg_probs = predict_with_tta(
                    model, img, device, 
                    num_augmentations=tta_num_aug,
                    img_size=config.IMG_SIZE
                )
                pred = avg_probs.argmax(dim=1).item()
                batch_preds.append(pred)
            
            all_preds.extend(batch_preds)
            all_filenames.extend(filenames)
    else:
        # Standard prediction (faster)
        with torch.no_grad():
            for images, filenames in tqdm(test_loader, desc="Generating predictions"):
                images = images.to(device)
                
                with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                    outputs = model(images)
                
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_filenames.extend(filenames)
    
    # Convert class numbers to labels
    # Default class mapping (standard for this competition)
    if class_to_label is None:
        class_to_label = {
            0: 'Triple negative',
            1: 'Luminal A',
            2: 'Luminal B',
            3: 'HER2(+)'
        }
    
    predictions = [class_to_label[pred] for pred in all_preds]
    
    # Create submission DataFrame
    submission_df = pd.DataFrame({
        'sample_index': all_filenames,
        'label': predictions
    })
    
    submission_df.to_csv(output_path, index=False)
    print(f"✅ Submission saved to {output_path}")
    print(f"   Rows: {len(submission_df)}")
    print(f"   Masks applied: {use_mask}")
    print(f"   TTA enabled: {use_tta}")
    if use_tta:
        print(f"   TTA augmentations: {tta_num_aug}")
    
    if class_to_label:
        print(f"\nLabel distribution:")
        print(submission_df['label'].value_counts())
    
    return submission_df


# ============================================================
# visualization.py
# ============================================================

"""
Visualization utilities for activation maps and attention mechanisms
"""
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import cv2


class ActivationMapExtractor:
    """Extract and visualize activation maps from model layers"""
    
    def __init__(self, model, target_layer=None):
        """
        Args:
            model: The neural network model
            target_layer: Specific layer to hook (if None, uses last conv layer)
        """
        self.model = model
        self.activations = []
        self.gradients = []
        self.hooks = []
        
        # Auto-detect last convolutional layer if not specified
        if target_layer is None:
            target_layer = self._find_last_conv_layer()
        
        self.target_layer = target_layer
        
        # Register hooks
        if target_layer is not None:
            self._register_hooks(target_layer)
    
    def _find_last_conv_layer(self):
        """Find the last convolutional layer in the model"""
        last_conv = None
        
        # Handle AttentionWrapper
        model_to_search = self.model.backbone if hasattr(self.model, 'backbone') else self.model
        
        for name, module in model_to_search.named_modules():
            if isinstance(module, torch.nn.Conv2d):
                last_conv = module
        
        return last_conv
    
    def _register_hooks(self, layer):
        """Register forward and backward hooks"""
        def forward_hook(module, input, output):
            self.activations.append(output.detach())
        
        def backward_hook(module, grad_input, grad_output):
            self.gradients.append(grad_output[0].detach())
        
        self.hooks.append(layer.register_forward_hook(forward_hook))
        self.hooks.append(layer.register_full_backward_hook(backward_hook))
    
    def remove_hooks(self):
        """Remove all hooks"""
        for hook in self.hooks:
            hook.remove()
        self.hooks = []
    
    def get_activation_map(self, image, target_class=None):
        """
        Generate activation map for an image
        
        Args:
            image: Input tensor (1, C, H, W) or (C, H, W)
            target_class: Target class for Grad-CAM (if None, uses predicted class)
        
        Returns:
            activation_map: 2D numpy array
            predicted_class: Predicted class index
        """
        self.model.eval()
        self.activations = []
        self.gradients = []
        
        # Ensure batch dimension
        if image.dim() == 3:
            image = image.unsqueeze(0)
        
        image = image.to(next(self.model.parameters()).device)
        image.requires_grad = True
        
        # Forward pass
        output = self.model(image)
        predicted_class = output.argmax(dim=1).item()
        
        # Use predicted class if not specified
        if target_class is None:
            target_class = predicted_class
        
        # Backward pass
        self.model.zero_grad()
        class_score = output[0, target_class]
        class_score.backward()
        
        # Get activation and gradient
        activation = self.activations[0]  # (1, C, H, W)
        gradient = self.gradients[0]  # (1, C, H, W)
        
        # Grad-CAM: weight channels by gradients
        weights = gradient.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
        cam = (weights * activation).sum(dim=1, keepdim=True)  # (1, 1, H, W)
        
        # ReLU and normalize
        cam = F.relu(cam)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        
        return cam, predicted_class
    
    def visualize_activation(self, image, original_image=None, target_class=None, 
                           alpha=0.4, cmap='jet', title=None):
        """
        Visualize activation map overlaid on original image
        
        Args:
            image: Input tensor (1, C, H, W) or (C, H, W) - preprocessed
            original_image: Original image tensor or numpy array (H, W, C) for overlay
            target_class: Target class for visualization
            alpha: Overlay transparency
            cmap: Colormap for heatmap
            title: Plot title
        
        Returns:
            fig: Matplotlib figure
        """
        # Get activation map
        cam, pred_class = self.get_activation_map(image, target_class)
        
        # Prepare original image for overlay
        if original_image is None:
            # Use preprocessed image (denormalize)
            img_display = image.squeeze().cpu().permute(1, 2, 0).numpy()
            # Simple denormalization
            img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min())
        else:
            if isinstance(original_image, torch.Tensor):
                img_display = original_image.cpu().numpy()
                if img_display.ndim == 4:
                    img_display = img_display[0]
                if img_display.shape[0] == 3:  # (C, H, W) -> (H, W, C)
                    img_display = img_display.transpose(1, 2, 0)
            else:
                img_display = original_image
            
            # Normalize to [0, 1]
            if img_display.max() > 1:
                img_display = img_display / 255.0
        
        # Resize activation map to match image size
        h, w = img_display.shape[:2]
        cam_resized = cv2.resize(cam, (w, h))
        
        # Create visualization
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # Original image
        axes[0].imshow(img_display)
        axes[0].set_title('Original Image')
        axes[0].axis('off')
        
        # Activation map
        im = axes[1].imshow(cam_resized, cmap=cmap)
        axes[1].set_title(f'Activation Map (Class {pred_class})')
        axes[1].axis('off')
        plt.colorbar(im, ax=axes[1])
        
        # Overlay
        axes[2].imshow(img_display)
        axes[2].imshow(cam_resized, cmap=cmap, alpha=alpha)
        axes[2].set_title('Overlay')
        axes[2].axis('off')
        
        if title:
            fig.suptitle(title, fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        return fig
    
    def __del__(self):
        """Cleanup hooks"""
        self.remove_hooks()


def visualize_attention_maps(model, dataloader, num_samples=5, class_names=None, device='cuda'):
    """
    Visualize attention maps for random samples from dataloader
    
    Args:
        model: Model with attention mechanism (AttentionWrapper)
        dataloader: DataLoader with images
        num_samples: Number of samples to visualize
        class_names: List of class names for labels
        device: Device to run on
    
    Returns:
        fig: Matplotlib figure
    """
    model.eval()
    
    # Check if model has attention
    if not hasattr(model, 'get_attention_map'):
        print("⚠️ Model does not have attention mechanism")
        return None
    
    # Get samples
    images, labels = next(iter(dataloader))
    images = images[:num_samples].to(device)
    labels = labels[:num_samples].cpu().numpy()
    
    # Forward pass to generate attention maps
    with torch.no_grad():
        outputs = model(images)
        predictions = outputs.argmax(dim=1).cpu().numpy()
    
    # Get attention maps
    attention_maps = model.get_attention_map()
    if attention_maps is None:
        print("⚠️ No attention maps available")
        return None
    
    # Visualize
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(num_samples):
        # Original image (denormalize)
        img = images[i].cpu().permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min())
        
        # Attention map
        attn = attention_maps[i, 0].cpu().numpy()
        attn_resized = cv2.resize(attn, (img.shape[1], img.shape[0]))
        
        # Plot
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f'Original\nTrue: {labels[i]}, Pred: {predictions[i]}')
        axes[i, 0].axis('off')
        
        im = axes[i, 1].imshow(attn_resized, cmap='jet')
        axes[i, 1].set_title('Attention Map')
        axes[i, 1].axis('off')
        plt.colorbar(im, ax=axes[i, 1])
        
        axes[i, 2].imshow(img)
        axes[i, 2].imshow(attn_resized, cmap='jet', alpha=0.5)
        axes[i, 2].set_title('Overlay')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    return fig


def debug_activations(model, image, layer_names=None):
    """
    Debug utility to inspect activations at multiple layers
    
    Args:
        model: Neural network model
        image: Input tensor
        layer_names: List of layer names to inspect (if None, shows all conv layers)
    
    Returns:
        activations_dict: Dictionary mapping layer names to activation tensors
    """
    model.eval()
    activations = {}
    
    def hook_fn(name):
        def hook(module, input, output):
            activations[name] = output.detach()
        return hook
    
    # Register hooks
    hooks = []
    model_to_search = model.backbone if hasattr(model, 'backbone') else model
    
    for name, module in model_to_search.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            if layer_names is None or name in layer_names:
                hooks.append(module.register_forward_hook(hook_fn(name)))
    
    # Forward pass
    with torch.no_grad():
        _ = model(image)
    
    # Remove hooks
    for hook in hooks:
        hook.remove()
    
    # Print summary
    print("\n📊 Activation Summary:")
    for name, act in activations.items():
        print(f"  {name}: {act.shape}")
    
    return activations


def plot_activation_statistics(activations_dict):
    """Plot statistics of activations across layers"""
    layer_names = list(activations_dict.keys())
    means = [activations_dict[name].mean().item() for name in layer_names]
    stds = [activations_dict[name].std().item() for name in layer_names]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Mean activations
    axes[0].bar(range(len(layer_names)), means)
    axes[0].set_xticks(range(len(layer_names)))
    axes[0].set_xticklabels(layer_names, rotation=45, ha='right')
    axes[0].set_title('Mean Activation per Layer')
    axes[0].set_ylabel('Mean Value')
    axes[0].grid(True, alpha=0.3)
    
    # Std activations
    axes[1].bar(range(len(layer_names)), stds)
    axes[1].set_xticks(range(len(layer_names)))
    axes[1].set_xticklabels(layer_names, rotation=45, ha='right')
    axes[1].set_title('Activation Std per Layer')
    axes[1].set_ylabel('Std Value')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig



# ============================================================
# data_cleaning.py
# ============================================================

"""
Data cleaning and preprocessing utilities
"""
import os
import shutil
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm


def create_clean_dataset(original_df, source_dir, dest_dir, dest_csv, threshold=0.005):
    """
    Filters the dataset and copies valid images/masks to a new folder.
    
    Args:
        original_df: Original DataFrame with labels
        source_dir: Source directory with train_data
        dest_dir: Destination directory for clean data
        dest_csv: Path to save cleaned CSV
        threshold: Minimum tissue percentage to keep sample
    
    Returns:
        DataFrame with cleaned data
    """
    print(f"🧹 Creating clean dataset in: {dest_dir}")
    
    # Create destination directories
    os.makedirs(dest_dir, exist_ok=True)
    os.makedirs(os.path.join(dest_dir, "images"), exist_ok=True)
    os.makedirs(os.path.join(dest_dir, "masks"), exist_ok=True)
    
    clean_data = []
    issues = []
    
    for idx, row in tqdm(original_df.iterrows(), total=len(original_df), desc="Cleaning data"):
        img_name = row['Image']
        mask_name = row['Mask']
        label = row['Label']
        
        # Build file paths
        img_path = os.path.join(source_dir, "images", img_name)
        mask_path = os.path.join(source_dir, "masks", mask_name)
        
        # Check if files exist
        if not os.path.exists(img_path):
            issues.append(f"Missing image: {img_name}")
            continue
        
        if not os.path.exists(mask_path):
            issues.append(f"Missing mask: {mask_name}")
            continue
        
        try:
            # Load and validate mask
            mask = np.array(Image.open(mask_path))
            
            # Calculate tissue percentage
            tissue_pixels = np.sum(mask > 0)
            total_pixels = mask.shape[0] * mask.shape[1]
            tissue_percentage = tissue_pixels / total_pixels
            
            # Filter out samples with too little tissue
            if tissue_percentage < threshold:
                issues.append(f"Low tissue ({tissue_percentage:.4f}): {img_name}")
                continue
            
            # Copy files to clean directory
            dest_img_path = os.path.join(dest_dir, "images", img_name)
            dest_mask_path = os.path.join(dest_dir, "masks", mask_name)
            
            shutil.copy2(img_path, dest_img_path)
            shutil.copy2(mask_path, dest_mask_path)
            
            # Add to clean data
            clean_data.append({
                'Image': img_name,
                'Mask': mask_name,
                'Label': label,
                'TissuePercentage': tissue_percentage
            })
            
        except Exception as e:
            issues.append(f"Error processing {img_name}: {str(e)}")
            continue
    
    # Create cleaned DataFrame
    clean_df = pd.DataFrame(clean_data)
    clean_df.to_csv(dest_csv, index=False)
    
    print(f"\n✅ Clean dataset created:")
    print(f"   Original samples: {len(original_df)}")
    print(f"   Clean samples: {len(clean_df)}")
    print(f"   Removed: {len(original_df) - len(clean_df)}")
    print(f"   Issues found: {len(issues)}")
    
    if issues:
        print(f"\n⚠️  First 10 issues:")
        for issue in issues[:10]:
            print(f"   - {issue}")
    
    return clean_df


def analyze_dataset(df, title="Dataset Analysis"):
    """Quick dataset analysis"""
    print(f"\n{'='*50}")
    print(f"{title}")
    print(f"{'='*50}")
    
    print(f"\nTotal samples: {len(df)}")
    print(f"\nClass distribution:")
    class_counts = df['Label'].value_counts().sort_index()
    for label, count in class_counts.items():
        print(f"   Class {label}: {count:4d} ({count/len(df)*100:5.2f}%)")
    
    if 'TissuePercentage' in df.columns:
        print(f"\nTissue coverage:")
        print(f"   Mean: {df['TissuePercentage'].mean():.4f}")
        print(f"   Min:  {df['TissuePercentage'].min():.4f}")
        print(f"   Max:  {df['TissuePercentage'].max():.4f}")



# ============================================================
# outlier_detection.py
# ============================================================

"""
Outlier detection and data cleaning utilities
Implements ADVICE 05/12 - Inspect Outliers
"""
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os
from tqdm import tqdm


def find_high_loss_samples(model, dataloader, device, top_k=100):
    """
    Find samples with highest loss values
    
    Args:
        model: Trained model
        dataloader: DataLoader with dataset to inspect
        device: cuda or cpu
        top_k: Number of highest loss samples to return
    
    Returns:
        List of (index, loss_value, predicted_label, true_label)
    """
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction='none')  # Per-sample loss
    
    losses = []
    predictions = []
    labels = []
    indices = []
    
    with torch.no_grad():
        for batch_idx, (images, batch_labels) in enumerate(tqdm(dataloader, desc="Computing losses")):
            images = images.to(device)
            batch_labels = batch_labels.to(device)
            
            outputs = model(images)
            batch_losses = criterion(outputs, batch_labels)
            _, preds = torch.max(outputs, 1)
            
            # Store results
            batch_size = images.size(0)
            batch_indices = range(batch_idx * dataloader.batch_size, 
                                 batch_idx * dataloader.batch_size + batch_size)
            
            losses.extend(batch_losses.cpu().numpy())
            predictions.extend(preds.cpu().numpy())
            labels.extend(batch_labels.cpu().numpy())
            indices.extend(batch_indices)
    
    # Create results dataframe
    results = pd.DataFrame({
        'index': indices,
        'loss': losses,
        'predicted': predictions,
        'true_label': labels,
        'correct': [p == l for p, l in zip(predictions, labels)]
    })
    
    # Sort by loss (highest first)
    results = results.sort_values('loss', ascending=False)
    
    return results.head(top_k)


def visualize_outliers(high_loss_df, dataset, save_path='outliers_visualization.png', 
                      num_samples=20):
    """
    Visualize high-loss samples to inspect mislabeled or corrupted data
    
    Args:
        high_loss_df: DataFrame from find_high_loss_samples()
        dataset: Original dataset to get images
        save_path: Where to save the visualization
        num_samples: Number of samples to visualize
    """
    num_samples = min(num_samples, len(high_loss_df))
    
    # Create grid
    cols = 5
    rows = (num_samples + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 3))
    if rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx, (_, row) in enumerate(high_loss_df.head(num_samples).iterrows()):
        r = idx // cols
        c = idx % cols
        ax = axes[r, c]
        
        # Get image
        image_idx = int(row['index'])
        image, true_label = dataset[image_idx]
        
        # Convert tensor to numpy for visualization
        if isinstance(image, torch.Tensor):
            image = image.permute(1, 2, 0).cpu().numpy()
            # Denormalize if needed (assuming ImageNet normalization)
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            image = std * image + mean
            image = np.clip(image, 0, 1)
        
        ax.imshow(image)
        
        # Add information
        loss = row['loss']
        pred = int(row['predicted'])
        true = int(row['true_label'])
        correct = row['correct']
        
        color = 'green' if correct else 'red'
        title = f"Loss: {loss:.3f}\nTrue: {true}, Pred: {pred}"
        ax.set_title(title, color=color, fontsize=10)
        ax.axis('off')
    
    # Hide unused subplots
    for idx in range(num_samples, rows * cols):
        r = idx // cols
        c = idx % cols
        axes[r, c].axis('off')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"✅ Outlier visualization saved to {save_path}")
    plt.close()


def detect_corrupted_images(data_dir, image_column='Image', min_size=32):
    """
    Detect corrupted or invalid images in dataset
    
    Args:
        data_dir: Directory containing images
        image_column: Column name with image filenames
        min_size: Minimum valid image size
    
    Returns:
        List of corrupted image paths
    """
    corrupted = []
    
    image_dir = os.path.join(data_dir, "images")
    if not os.path.exists(image_dir):
        print(f"⚠️  Directory not found: {image_dir}")
        return corrupted
    
    image_files = os.listdir(image_dir)
    
    for img_file in tqdm(image_files, desc="Checking images"):
        img_path = os.path.join(image_dir, img_file)
        
        try:
            img = Image.open(img_path)
            img.verify()  # Check if file is corrupted
            
            # Re-open after verify (verify closes the file)
            img = Image.open(img_path)
            
            # Check size
            if img.size[0] < min_size or img.size[1] < min_size:
                corrupted.append(img_path)
                print(f"⚠️  Too small: {img_file} ({img.size})")
                continue
            
            # Check mode
            if img.mode not in ['RGB', 'L']:
                corrupted.append(img_path)
                print(f"⚠️  Invalid mode: {img_file} ({img.mode})")
                continue
                
        except Exception as e:
            corrupted.append(img_path)
            print(f"❌ Corrupted: {img_file} - {str(e)}")
    
    return corrupted


def find_label_errors(high_loss_df, threshold_percentile=95):
    """
    Find potential label errors based on consistently high loss
    
    Args:
        high_loss_df: DataFrame from find_high_loss_samples()
        threshold_percentile: Loss percentile threshold for suspicious samples
    
    Returns:
        DataFrame with suspicious samples
    """
    threshold = np.percentile(high_loss_df['loss'], threshold_percentile)
    
    suspicious = high_loss_df[
        (high_loss_df['loss'] > threshold) & 
        (~high_loss_df['correct'])
    ]
    
    print(f"\n🔍 Found {len(suspicious)} potentially mislabeled samples")
    print(f"   Loss threshold: {threshold:.3f} ({threshold_percentile}th percentile)")
    
    return suspicious


def create_outlier_report(model, dataloader, dataset, device, output_dir='outlier_analysis'):
    """
    Complete outlier analysis pipeline
    
    Args:
        model: Trained model
        dataloader: DataLoader
        dataset: Original dataset
        device: cuda or cpu
        output_dir: Directory to save results
    
    Returns:
        Dictionary with analysis results
    """
    os.makedirs(output_dir, exist_ok=True)
    
    print("🔍 Starting outlier analysis...")
    
    # 1. Find high loss samples
    print("\n1️⃣  Finding high-loss samples...")
    high_loss_df = find_high_loss_samples(model, dataloader, device, top_k=200)
    
    # 2. Detect label errors
    print("\n2️⃣  Detecting potential label errors...")
    suspicious = find_label_errors(high_loss_df, threshold_percentile=95)
    
    # 3. Visualize outliers
    print("\n3️⃣  Visualizing outliers...")
    viz_path = os.path.join(output_dir, 'high_loss_samples.png')
    visualize_outliers(high_loss_df, dataset, save_path=viz_path, num_samples=20)
    
    # 4. Save reports
    print("\n4️⃣  Saving reports...")
    high_loss_df.to_csv(os.path.join(output_dir, 'high_loss_samples.csv'), index=False)
    suspicious.to_csv(os.path.join(output_dir, 'suspicious_labels.csv'), index=False)
    
    # 5. Generate summary
    summary = {
        'total_samples': len(high_loss_df),
        'suspicious_labels': len(suspicious),
        'avg_loss_top100': high_loss_df.head(100)['loss'].mean(),
        'accuracy_top100': high_loss_df.head(100)['correct'].mean(),
        'suspicious_indices': suspicious['index'].tolist()
    }
    
    print("\n" + "="*50)
    print("📊 OUTLIER ANALYSIS SUMMARY")
    print("="*50)
    print(f"Total high-loss samples analyzed: {summary['total_samples']}")
    print(f"Potentially mislabeled samples: {summary['suspicious_labels']}")
    print(f"Average loss (top 100): {summary['avg_loss_top100']:.3f}")
    print(f"Accuracy on top 100 highest-loss: {summary['accuracy_top100']:.1%}")
    print(f"\n💾 Results saved to: {output_dir}/")
    print("="*50)
    
    return summary


print('✅ All modules loaded')

In [ ]:
# Google Colab setup (uncomment if needed)
# from google.colab import drive
# drive.mount("/gdrive")
# %cd "/gdrive/My Drive/Challenge2"

## ⚙️ Setup

In [ ]:
# Install dependencies if needed
# !pip install pyyaml tensorboard scikit-learn pandas matplotlib seaborn tqdm

In [ ]:
import sys
import os
from pathlib import Path

# Add utils to path
sys.path.insert(0, str(Path.cwd()))

# Import everything from our streamlined package
# from utils import (
#     Config, set_seed, setup_device, create_dirs,
#     get_transforms, create_dataloaders,
#     get_model, Trainer,
#     evaluate_model, create_submission,
#     plot_training_history, plot_confusion_matrix
# )

import pandas as pd
import torch
from sklearn.model_selection import train_test_split

print("✅ All imports successful")

ModuleNotFoundError: No module named 'cv2'

## 📝 Configuration

**SINGLE SOURCE OF TRUTH** - Edit `config.yaml` or modify Config here

In [ ]:
# Option 1: Load from YAML file (recommended)
try:
    config = Config.from_yaml('config.yaml')
    print("✅ Loaded config from config.yaml")
except:
    print("⚠️  config.yaml not found, using defaults")
    config = Config()

# Option 2: Override specific settings
# config.NUM_EPOCHS = 30
# config.BATCH_SIZE = 32
# config.LEARNING_RATE = 0.001
# config.MODEL_NAME = 'resnet50'
# config.USE_TENSORBOARD = True

# Save current config for reproducibility
config.save_yaml('current_run_config.yaml')

print(f"\n📋 Config:")
print(f"   Model: {config.MODEL_NAME}")
print(f"   Epochs: {config.NUM_EPOCHS}")
print(f"   Batch: {config.BATCH_SIZE}")
print(f"   LR: {config.LEARNING_RATE}")

## 🎲 Reproducibility

In [ ]:
set_seed(config.SEED)
device = setup_device()
create_dirs(config)

print(f"✅ Seed: {config.SEED} | Device: {device}")

## 📊 Data Preparation

In [ ]:
# Load CSV
df = pd.read_csv(config.CSV_PATH)
print(f"✅ Loaded {len(df)} samples from {config.CSV_PATH}")

# Split train/val
train_df, val_df = train_test_split(
    df, 
    test_size=config.VAL_SPLIT,
    stratify=df['label'],
    random_state=config.SEED
)

print(f"   Train: {len(train_df)} | Val: {len(val_df)}")
print(f"\nClass distribution:")
print(train_df['label'].value_counts().sort_index())

In [ ]:
# Create data loaders
train_loader, val_loader, num_classes = create_dataloaders(
    train_df=train_df,
    val_df=val_df,
    data_dir=config.DATA_DIR,
    config=config,
    use_imagenet_norm=config.USE_IMAGENET_NORM
)

print(f"✅ Dataloaders created")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")
print(f"   Number of classes detected: {num_classes}")

## 🏗️ Model

In [ ]:
# Build model
config.NUM_CLASSES = num_classes  # Use detected number of classes
model = get_model(config)
model = model.to(device)

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model: {config.MODEL_NAME}")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

## 🚀 Training

TensorBoard will log automatically if `USE_TENSORBOARD=True`

In [ ]:
# Start TensorBoard in background
if config.USE_TENSORBOARD:
    %load_ext tensorboard
    %tensorboard --logdir {config.TENSORBOARD_DIR}

In [ ]:
# Create trainer - all settings from config
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device
)

# Train (supports auto-resume)
history = trainer.train(resume=False)

print("\n✅ Training complete!")

## 📈 Results

In [ ]:
# Plot training curves
history_df = trainer.get_history_df()
display(history_df.tail())

plot_training_history(history_df)

In [ ]:
# Evaluate on validation set
results = evaluate_model(
    model, 
    val_loader, 
    device,
    class_names=[f'Class_{i}' for i in range(num_classes)]
)

print(f"\n📊 Validation Results:")
print(f"   Accuracy: {results['accuracy']:.4f}")
print(f"   F1 Score: {results['f1']:.4f}")

# Confusion matrix
plot_confusion_matrix(
    results['confusion_matrix'],
    class_names=[f'Class_{i}' for i in range(num_classes)]
)

## 💾 Create Submission

In [ ]:
# Load best model
exp_name = config.EXPERIMENT_NAME
best_model_path = f"{config.MODELS_DIR}/{exp_name}_best.pt"
model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))
print(f"✅ Loaded best model from {best_model_path}")

# Create submission (with masks if available)
submission_df = create_submission(
    model=model,
    test_dir=config.TEST_DIR,
    config=config,
    device=device,
    output_path='submission.csv',
    use_mask=True  # Apply masks to test images if available
)

print(f"✅ Submission saved to submission.csv")

---

## 🔧 Advanced: Two-Stage Training

Optionally train in two stages: frozen backbone first, then fine-tune

In [ ]:
# Stage 1: Train classifier only (frozen backbone)
config.FREEZE_BACKBONE = True
config.NUM_EPOCHS = 10
config.LEARNING_RATE = 0.001

model_stage1 = get_model(config)
trainer_stage1 = Trainer(model_stage1, train_loader, val_loader, config, device)
history_stage1 = trainer_stage1.train()

print("✅ Stage 1 complete (frozen backbone)")

In [ ]:
# Stage 2: Fine-tune entire model
from utils import freeze_backbone

freeze_backbone(model_stage1, freeze=False)  # Unfreeze
config.NUM_EPOCHS = 20
config.LEARNING_RATE = 0.0001  # Lower LR for fine-tuning

trainer_stage2 = Trainer(model_stage1, train_loader, val_loader, config, device)
history_stage2 = trainer_stage2.train()

print("✅ Stage 2 complete (fine-tuned)")